# Reducing Hospital Costs Through ML-Driven Diabetic Readmission Prevention

**Author: Nikhitha Jella | Data Professional | nikhithajvv3@gmail.com**


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


## **Data Loading**


In [ ]:
df = pd.read_csv(r'Data\diabetic_data.csv')
id_lookup = pd.read_csv(r'Data\IDS_mapping.csv')



Two source datasets are loaded:
- `diabetic_data.csv`: Core patient encounter records — includes admission details, lab results, medication regimens, and readmission outcomes across 130 U.S. hospitals.
- `IDS_mapping.csv`: Lookup table for decoding numeric identifiers (admission type, discharge disposition, admission source) into human-readable labels.

Both datasets together form the raw foundation for building the cost-risk prediction pipeline.


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


**Initial Dataset Assessment**

A high-level scan of the dataset reveals:

**Key Observations**:
- **Scale**: 101,766 patient encounters, 50 feature columns — large enough for reliable model training
- **Data Types**: 13 numeric columns, 37 object (categorical/string) columns
- **Critical Gaps**:
  - `max_glu_serum`: Present for only 5.3% of records — most patients never had this test ordered
  - `A1Cresult`: Available for 16.7% of patients — limited but still usable as a binary signal
  - Other columns appear structurally complete but contain `"?"` as a missing value placeholder


## **Data Cleaning**


In [ ]:
admin_cols = ['encounter_id', 'patient_nbr']
df.drop(columns=admin_cols, inplace=True)



`encounter_id` and `patient_nbr` are administrative identifiers — they uniquely tag each row but carry zero signal about readmission risk. Including them would only risk model overfitting to arbitrary record IDs.


In [ ]:
df[['race', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']].head()



Missing data in this dataset is encoded as `"?"` rather than standard `NaN`. This is a common pattern in older EHR exports and must be normalized before any analysis or modeling.


In [ ]:
placeholder = '?'
for col in df.columns:
    flagged = df[col].isin([placeholder]).sum()
    if flagged > 0:
        print(f"{col}: {flagged} placeholder values found")


In [ ]:
df.replace('?', np.nan, inplace=True)
df.isnull().sum()


In [ ]:
df[['A1Cresult', 'max_glu_serum']].head()


In [ ]:
df.fillna('none', inplace=True)
df[['A1Cresult', 'max_glu_serum']].head()



After converting `"?"` to `NaN`, remaining nulls are filled with `"none"` — a deliberate choice for clinical columns like `A1Cresult` and `max_glu_serum`.

Here, `"none"` is meaningful: it signals the test was not ordered, which itself carries cost-relevant information (patients without glucose monitoring may have higher undetected instability at discharge).


#### Targeted Fill Strategy and Column Removal


In [ ]:
fill_map = {
    'medical_specialty': 'Not Recorded',
    'payer_code': 'Unknown',
    'race': 'Not Specified'
}
for col, val in fill_map.items():
    df[col] = df[col].fillna(val)

df.drop(columns=['weight'], inplace=True)

for diag_col in ['diag_1', 'diag_2', 'diag_3']:
    df[diag_col] = df[diag_col].fillna('Missing')

df.isnull().sum()



Each missing field is handled according to its clinical role:
- `medical_specialty`: Filled with `'Not Recorded'` — specialty data is incomplete but not a reason to drop the column
- `payer_code`: Filled with `'Unknown'` — payer type has mild cost-pathway relevance
- `race`: Filled with `'Not Specified'` — retained for demographic analysis
- `diag_1`, `diag_2`, `diag_3`: Filled with `'Missing'` — explicit label to distinguish from valid ICD-9 codes
- `weight`: Dropped entirely — over 97% missing, imputation would introduce more noise than signal


In [ ]:
df.head(10)


### Missing Value Handling — Summary

Steps taken to normalize the dataset for downstream use:
- Identified `"?"` as the dataset's missing value convention; converted to `NaN`
- Applied `"none"` for untested lab columns (`A1Cresult`, `max_glu_serum`) — absence of a test is its own clinical signal
- Context-specific fills:
  - `race` → `"Not Specified"`
  - `payer_code` → `"Unknown"`
  - `medical_specialty` → `"Not Recorded"`
  - `diag_1` to `diag_3` → `"Missing"`
- Dropped `weight` — too sparse to be recoverable


## **Feature Engineering**


**ICD-9 Diagnosis Code Grouping**


In [ ]:
def categorize_icd9(code):
    if code == 'Missing':
        return 'Missing'
    if isinstance(code, str):
        code = code.strip().upper()
        if code.startswith('V'):
            return 'V code (Supplementary Factors)'
        elif code.startswith('E'):
            return 'E code (External Causes)'
    try:
        num = int(float(code))
        ranges = [
            (1,   139,  'Infectious'),
            (140, 239,  'Neoplasms (Tumors, Cancer)'),
            (240, 279,  'Endocrine/Metabolic'),
            (280, 289,  'Blood Diseases'),
            (290, 319,  'Mental Disorders'),
            (320, 389,  'Nervous System'),
            (390, 459,  'Circulatory System'),
            (460, 519,  'Respiratory System'),
            (520, 579,  'Digestive System'),
            (580, 629,  'Genitourinary System'),
            (630, 679,  'Pregnancy Complications'),
            (680, 709,  'Skin and Subcutaneous Tissue'),
            (710, 739,  'Musculoskeletal System'),
            (740, 759,  'Congenital Anomalies'),
            (760, 779,  'Perinatal Conditions'),
            (780, 799,  'Symptoms and Undefined Conditions'),
            (800, 999,  'Injury and Poisoning'),
        ]
        for lo, hi, label in ranges:
            if lo <= num <= hi:
                return label
        return 'Other'
    except:
        return 'Other'


Raw ICD-9 codes are highly granular — thousands of unique values that would explode one-hot encoding and obscure the patterns that matter.

The custom `icd_9()` function maps these codes into 12 clinical categories (e.g., Circulatory, Endocrine, Respiratory, Neoplasms). This:
- Reduces dimensionality dramatically
- Surfaces cost-relevant clusters (circulatory and endocrine conditions drive the highest readmission costs)
- Makes model outputs interpretable to clinical stakeholders


**High-Cost Diabetes Risk Flagging**


In [ ]:
severe_diabetes_codes = ['250.6', '250.7', '250.8', '250.9']
df['high_risk_diabetes'] = df['diag_1'].astype(str).isin(severe_diabetes_codes).astype(int)

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(categorize_icd9)



Specific ICD-9 subcodes (`250.6`, `250.7`, `250.8`, `250.9`) mark the most severe diabetic complications — peripheral circulatory disorders, neurological manifestations, and "other specified" complications. These are associated with the highest downstream care costs.

A binary flag `high_risk_diabetes` is created for patients with these codes in their primary diagnosis. The `icd_9()` function then converts all three diagnosis columns into grouped clinical categories for broader risk profiling.


In [ ]:
df.head(10)


In [ ]:
df['high_risk_diabetes'].value_counts()


**Diagnosis Severity Scoring**


In [ ]:
for col in ['diag_1', 'diag_2', 'diag_3']:
    print(f"{col} categories:", df[col].unique())


In [ ]:
diagnosis_risk = {
    'Endocrine/Metabolic': 1,
    'Pregnancy Complications': 1,
    'Infectious': 1,
    'Neoplasms (Tumors, Cancer)': 1,
    'Circulatory System': 1,
    'Respiratory System': 1,
    'Injury and Poisoning': 1,
    'Mental Disorders': 1,
    'Nervous System': 1,
    'Blood Diseases': 1,
    'Congenital Anomalies': 1,
    'E code (External Causes)': 1,
    'Skin and Subcutaneous Tissue': 0,
    'Musculoskeletal System': 0,
    'Digestive System': 0,
    'V code (Supplementary Factors)': 0,
    'Symptoms and Undefined Conditions': 0,
    'Genitourinary System': 0,
    'Missing': 0,
    'Perinatal Conditions': 0,
}

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].map(diagnosis_risk)


Each clinical category is assigned a binary risk score based on expected readmission and cost burden:

- `1` (high-risk): Endocrine/Metabolic, Circulatory, Neoplasms, Infectious, Pregnancy Complications
- `0` (lower-risk): categories with weaker readmission signal

This scoring converts multi-value categorical diagnoses into a single risk dimension usable by any classifier.


**Composite Diagnosis Risk Indicator**


In [ ]:
df['any_high_risk_diagnosis'] = (
    (df['diag_1'] == 1.0) | (df['diag_2'] == 1.0) | (df['diag_3'] == 1.0)
).astype(int)


The `any_high_risk_diagnosis` feature checks whether **any** of the three diagnosis slots carries a high-risk category:

- `1` if at least one of `diag_1`, `diag_2`, or `diag_3` is flagged high-risk
- `0` only when all three are low-risk

Patients admitted with comorbidities across multiple high-risk categories are systematically more expensive to treat and more likely to be readmitted — this feature captures that compound exposure.


Diagnosis column roles:
- `diag_1` → Primary diagnosis driving the admission
- `diag_2`, `diag_3` → Secondary and tertiary conditions active during the stay


In [ ]:
df.drop(columns=['diag_1', 'diag_2', 'diag_3'], inplace=True, errors='ignore')



After extracting clinical categories and risk scores from the raw ICD-9 codes, the original `diag_1`, `diag_2`, and `diag_3` columns are dropped.

The grouped features carry all the predictive signal with far less noise and dimensionality.


In [ ]:
df.head()


**Lab Result Encoding**


In [ ]:
glucose_scale = {'none': 0, 'norm': 1, '>200': 2, '>300': 3}
a1c_scale     = {'none': 0, 'norm': 1, '>7': 2,  '>8': 3}

df['max_glu_serum'] = df['max_glu_serum'].str.lower().map(glucose_scale)
df['A1Cresult']     = df['A1Cresult'].str.lower().map(a1c_scale)


Both lab columns are converted from string categories to ordered numeric scales:

- `max_glu_serum`: `none=0`, `norm=1`, `>200=2`, `>300=3`
- `A1Cresult`: `none=0`, `norm=1`, `>7=2`, `>8=3`

Higher values indicate worse glycemic control at the time of admission. Patients discharged with poorly controlled blood glucose are at measurable higher risk of readmission — and the cost of managing a preventable hypoglycemic or hyperglycemic crisis in the ER far exceeds the cost of pre-discharge intervention.


In [ ]:
df.head()


In [ ]:
df['A1Cresult'].value_counts().sort_index()


In [ ]:
df['max_glu_serum'].value_counts().sort_index()


#### Lab Signal Encoding — Summary

- `A1Cresult` and `max_glu_serum` converted to 4-level ordinal scales (0–3)
- The `0` level (test not ordered) itself carries information — clinicians ordering fewer tests may signal lower acuity or gaps in monitoring
- No column drops needed here; both features are retained in final model


**Healthcare Utilization Feature Construction**


In [ ]:
df['number_inpatient'].value_counts().head(10)


In [ ]:
df['number_emergency'].value_counts().head(10)


In [ ]:
df['number_outpatient'].value_counts().head(10)


In [ ]:
df['inpatient_category'] = pd.cut(
    df['number_inpatient'],
    bins=[-1, 0, 1, 3, float('inf')],
    labels=['None', 'One', 'Few', 'Many']
)
df['had_outpatient'] = (df['number_outpatient'] > 0).astype(int)



Prior healthcare utilization is one of the strongest predictors of future cost burden:

- `inpatient_category`: Buckets prior inpatient visits into `None / One / Few / Many` — repeat inpatient users are the highest-cost readmission segment
- `had_outpatient`: Binary flag for any prior outpatient visit — patients with outpatient engagement may have better care continuity

These features translate raw utilization counts into interpretable, model-friendly categories.



Utilization variables available in the dataset:

- `number_inpatient`: Prior inpatient admissions in the past year — strongest readmission cost predictor
- `number_emergency`: Prior ER visits — signals instability and reactive care-seeking behavior
- `number_outpatient`: Prior outpatient visits — signals care engagement and chronic disease management


The `inpatient_category` binning logic:

- `None`: 0 prior inpatient visits (baseline)
- `One`: Exactly 1 prior visit
- `Few`: 2–3 prior visits
- `Many`: 4+ prior visits — this tier represents the high-cost "frequent flier" population

The lower bound of -1 ensures zero is cleanly captured in the `None` bucket without floating-point edge cases.


In [ ]:
df.drop(columns=['number_outpatient'], inplace=True)


`number_outpatient` is redundant once `had_outpatient` (binary) is in place. Retaining both would introduce multicollinearity without adding new information to the model.


**Admission and Discharge Pathway Encoding**


In [ ]:
id_lookup.head()


In [ ]:
admission_flags = {
    'emergency_admission': 1,
    'urgent_admission':    2,
    'elective_admission':  3,
    'newborn_admission':   4,
    'trauma_admission':    7,
}
for col_name, type_id in admission_flags.items():
    df[col_name] = (df['admission_type_id'] == type_id).astype(int)


The `admission_type_id` integer codes are decoded into five binary flags:
- `emergency_admission`: Unplanned urgent care — these patients have less preparation for discharge and higher readmission rates
- `urgent_admission`: Clinically urgent but not life-threatening
- `elective_admission`: Scheduled — typically lower readmission risk
- `newborn_admission`: Specific neonatal pathway
- `trauma_admission`: Injury-driven admission

Emergency admissions in particular are associated with higher post-discharge instability and preventable return visits.


In [ ]:
df['discharged_home']            = (df['discharge_disposition_id'] == 1).astype(int)
df['discharged_to_facility']     = df['discharge_disposition_id'].isin([2,3,4,5,22,23,24,27,28,29,30]).astype(int)
df['discharged_with_home_health']= df['discharge_disposition_id'].isin([6, 8]).astype(int)
df['left_against_advice']        = (df['discharge_disposition_id'] == 7).astype(int)
df['readmitted_same_hospital']   = (df['discharge_disposition_id'] == 9).astype(int)
df['neonate_transfer']           = (df['discharge_disposition_id'] == 10).astype(int)
df['patient_expired']            = df['discharge_disposition_id'].isin([11,19,20,21]).astype(int)
df['still_patient']              = (df['discharge_disposition_id'] == 12).astype(int)
df['discharged_to_hospice']      = df['discharge_disposition_id'].isin([13,14]).astype(int)
df['discharged_within_institution'] = (df['discharge_disposition_id'] == 15).astype(int)
df['discharged_for_outpatient']  = df['discharge_disposition_id'].isin([16,17]).astype(int)



Discharge destination is one of the highest-signal features for readmission cost risk:

- `discharged_home`: Patient returns home — higher risk if inadequate home support
- `discharged_to_facility`: Transferred to SNF, rehab, or care facility — structured post-acute care lowers readmission rates
- `patient_expired`: Mortality during encounter
- `discharged_to_hospice`: End-of-life care pathway

The gap between `discharged_home` (higher risk) and `discharged_to_facility` (lower risk) is a key cost-containment lever — identifying home-discharge patients who need more support is a direct application of this model.


In [ ]:
df['provider_referral']     = df['admission_source_id'].isin([1, 2, 3]).astype(int)
df['transfer_from_facility']= df['admission_source_id'].isin([4, 5, 6]).astype(int)
df['from_emergency_room']   = (df['admission_source_id'] == 7).astype(int)
df['from_law_enforcement']  = (df['admission_source_id'] == 8).astype(int)


How the patient arrived at the hospital shapes the clinical context and predicts downstream costs:

- `provider_referral`: Referred by physician or clinic — suggests active care management, typically lower readmission risk
- `transfer_from_facility`: Arrived from another health facility — often higher acuity, higher cost
- `from_emergency_room`: Walked into ER — reactive care-seeking, historically highest readmission predictor in this segment
- `from_law_enforcement`: Edge case; minimal volume


In [ ]:
df.head()


In [ ]:
df.drop(columns=['admission_type_id', 'discharge_disposition_id', 'admission_source_id'],
        inplace=True, errors='ignore')



After extracting all admission/discharge pathway signals into interpretable binary features, the original coded columns (`admission_type_id`, `discharge_disposition_id`, `admission_source_id`) are dropped — they've been fully decoded and are no longer needed.


In [ ]:
df['diabetesMedication'] = (df['diabetesMed'] == 'Yes').astype(int)


`diabetesMedication` is a binary indicator:

- `1` if the patient was on any diabetes-specific medication during the encounter
- `0` otherwise

Patients on diabetes medications represent an actively managed but clinically complex population — any disruption to their regimen post-discharge carries elevated readmission and cost risk.


In [ ]:
insulin_states = {'Steady': 'insulin_steady', 'Up': 'insulin_up',
                  'Down': 'insulin_down', 'No': 'insulin_none'}
for state, col_name in insulin_states.items():
    df[col_name] = (df['insulin'] == state).astype(int)


The `insulin` column is decomposed into four binary features:

- `insulin_steady`: Dose maintained — stable management
- `insulin_up`: Dose increased — glycemic instability detected, higher post-discharge risk
- `insulin_down`: Dose reduced — may signal improvement or risk of underdosing
- `insulin_none`: No insulin prescribed

Insulin dose changes during a hospital stay are a strong signal that glucose management was not stabilized before discharge.


In [ ]:
df.drop(columns=['diabetesMed', 'insulin'], inplace=True, errors='ignore')



Original columns are removed after their information has been fully encoded into binary features:

- `diabetesMed` → replaced by `diabetesMedication`
- `insulin` → replaced by `insulin_steady`, `insulin_up`, `insulin_down`, `insulin_none`


### Target Variable


In [ ]:
df['readmitted_within_30'] = (df['readmitted'] == '<30').astype(int)



The binary target `readmitted_within_30` is created:

- `1`: Patient was readmitted within 30 days — the costly outcome this model aims to prevent
- `0`: Patient was not readmitted within 30 days (includes both no readmission and readmissions after 30 days)

The 30-day window is the industry standard for CMS penalty calculations and hospital quality benchmarks.


In [ ]:
df.drop(columns=['readmitted'], inplace=True, errors='ignore')


After encoding the 30-day readmission flag, the original multi-class `readmitted` column (`<30`, `>30`, `NO`) is dropped — the model only needs the binary outcome.


In [ ]:
df.head()


In [ ]:
age_bins = {
    '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3, '[40-50)': 4,
    '[50-60)': 5, '[60-70)': 6, '[70-80)': 7, '[80-90)': 8, '[90-100)': 9
}
df['age_ordinal'] = df['age'].map(age_bins)


#### Age → Ordinal Encoding

The `age` column uses decade bins (`[0-10)`, `[10-20)`, ..., `[90-100)`), which are converted to ordinal integers 0–9.

This preserves the natural ordering of age while making it compatible with numerical models. Age is a key factor in readmission cost: elderly patients (age bins 6+) have both higher readmission rates and significantly higher per-episode costs.


Converting `inpatient_category` to an ordinal scale and creating an interaction term:
- Ordinal encoding: `None=0, One=1, Few=2, Many=3`
- Interaction term: `age_ordinal × inpatient_ordinal` — captures the compound risk of older patients with heavy prior utilization


In [ ]:
visit_frequency_map = {'None': 0, 'One': 1, 'Few': 2, 'Many': 3}
df['inpatient_ordinal'] = df['inpatient_category'].astype(str).map(visit_frequency_map)
df['age_inpatient_interaction'] = df['age_ordinal'] * df['inpatient_ordinal']


#### Utilization-Age Interaction Feature

- `inpatient_category` is mapped to `inpatient_ordinal` (None=0, One=1, Few=2, Many=3)
- `age_inpatient_interaction` = `age_ordinal × inpatient_ordinal`

This interaction captures a well-known pattern in healthcare cost data: a 70-year-old with 5 prior admissions carries fundamentally different risk than a 30-year-old with the same utilization history.


**Medication Complexity Encoding**


In [ ]:
diabetes_meds = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]
for med in diabetes_meds:
    if med in df.columns:
        df[f'{med}_used'] = (df[med] != 'No').astype(int)


Binary Medication Usage Features

For each of the 22 diabetes medication columns, a binary indicator (`{med}_used`) is created:
- `1` if the medication was prescribed, adjusted, or continued (any value other than `'No'`)
- `0` if not prescribed

Medication complexity is a critical cost driver — patients on multiple agents have higher adherence burden post-discharge and greater risk of adverse events that trigger costly returns.


In [ ]:
med_flag_cols = [f'{m}_used' for m in diabetes_meds if f'{m}_used' in df.columns]
df['medications_used'] = df[med_flag_cols].sum(axis=1)


`medications_used` aggregates all binary medication indicators into a total count of active medications per patient.

Higher medication counts signal clinical complexity and correlate with post-discharge management difficulty — a key cost risk factor.


In [ ]:
if 'change' in df.columns:
    df['medication_changed'] = (df['change'] == 'Ch').astype(int)


`medication_changed` is a binary flag indicating whether a medication adjustment (`'Ch'`) occurred during the encounter.

Mid-stay medication changes suggest the patient's condition was not stable at admission — and potentially still in flux at discharge.


In [ ]:
df.drop(columns=med_flag_cols, inplace=True, errors='ignore')



After engineering medication complexity features, the raw medication columns are dropped to eliminate redundancy. The model will use the engineered signals rather than the original multi-value strings.


**Medication × Diagnosis Severity Interaction**


In [ ]:
df['high_risk_with_medication'] = df['high_risk_diabetes'] * df['diabetesMedication']


The `high_risk_with_medication` feature is the product of:
- `high_risk_diabetes`: Flag for severe diabetic diagnosis codes
- `diabetesMedication`: Flag for active diabetes medication use

Patients with both high-risk diagnosis AND active medication management represent a segment where discharge instability is most likely to convert into a costly readmission. This interaction term helps the model capture that compounding risk.


In [ ]:
cols_to_remove = [m for m in diabetes_meds if m in df.columns]
if 'change' in df.columns:
    cols_to_remove.append('change')
df.drop(columns=cols_to_remove, inplace=True, errors='ignore')


Final cleanup removes:
- All individual `{med}_used` binary columns (captured in `medications_used` total count)
- The original `change` column (replaced by `medication_changed`)

Only the highest-signal engineered features are retained going forward.


## **Exploratory Data Analysis (EDA)**


### Univariate Analysis — Understanding Each Feature Independently


In [ ]:
target_dist = df['readmitted_within_30'].value_counts()
print(target_dist)


Analyzing the distribution of the target variable `readmitted_within_30`:

- **Raw counts**: How many patients fall in each class
- **Class percentages**: The proportion of costly readmissions in the dataset

This baseline distribution determines the severity of class imbalance and sets the benchmark for model performance — a naive model predicting all zeros would hit ~88.9% accuracy without catching a single readmission.


In [ ]:
target_pct = df['readmitted_within_30'].value_counts(normalize=True).mul(100).round(2)
print(target_pct)


In [ ]:
same_hosp = df['readmitted_same_hospital'].value_counts()
print(same_hosp)


In [ ]:
same_hosp_pct = df['readmitted_same_hospital'].value_counts(normalize=True).mul(100).round(2)
print(same_hosp_pct)


Examining `readmitted_same_hospital` separately to understand what fraction of 30-day readmissions return to the same facility.

Same-hospital readmissions are particularly relevant for hospital cost accountability — CMS readmission penalties apply regardless of where the patient returns, but same-facility readmissions are the clearest signal of discharge process failure.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x=df['readmitted_within_30'], palette=['#4e9af1', '#f45c51'], ax=ax)
ax.set_title('30-Day Readmission Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Readmitted Within 30 Days')
ax.set_ylabel('Patient Count')
ax.set_xticks([0, 1])
ax.set_xticklabels(['No Readmission', 'Readmitted'])
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()


The count plot shows the raw class distribution for `readmitted_within_30`:

- **0 (No readmission)**: The dominant class (~88.9% of encounters)
- **1 (Readmitted within 30 days)**: The costly minority class we need to predict

The visual imbalance is stark — this is why accuracy alone is a misleading metric for this problem.



Key baseline statistics:
- **11.1%** of patients are readmitted within 30 days
- **~0.02%** of those return to the same hospital

An 11.1% readmission rate across 100K encounters represents thousands of costly events — even a modest improvement in early identification translates to significant cost avoidance at scale.



The 9:1 class ratio will suppress minority class detection in standard models. Mitigation strategies (class weighting, SMOTE, threshold tuning) are explored in the modeling section.


**Demographics**


**Age Distribution**


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(x=df['age_ordinal'], palette='Blues_d', ax=ax)
ax.set_title('Patient Age Group Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Age Group (0=0-10yrs ... 9=90-100yrs)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


The age distribution reveals a dataset weighted toward middle-aged to elderly patients — consistent with a diabetic patient population.

From a cost perspective, the 60–80 age range (ordinal 6–8) is the highest-cost segment: these patients have more comorbidities, longer stays, and higher readmission rates than younger cohorts.


**Gender Distribution**


In [ ]:
gender_dist = df['gender'].value_counts()
print(gender_dist)


In [ ]:
invalid_mask = df['gender'].isin(['Unknown/Invalid'])
print(f"Invalid gender records: {invalid_mask.sum()}")
df[invalid_mask].head()



The `"Unknown/Invalid"` gender category represents a negligible number of records with no clinical interpretability. Removing them ensures the gender feature contributes clean signal rather than noise to the model.


In [ ]:
df = df[~invalid_mask].copy()


In [ ]:
gender_dist = df['gender'].value_counts()
print(gender_dist)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(gender_dist, labels=gender_dist.index, autopct='%1.1f%%',
       startangle=90, colors=['#4e9af1', '#f9a8d4'])
ax.set_title('Gender Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Race Distribution**


In [ ]:
race_dist = df['race'].value_counts()
print(race_dist)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x=df['race'], order=race_dist.index, palette='Set1', ax=ax)
ax.set_title('Race Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Race')
ax.set_ylabel('Count')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


**Healthcare Utilization — Visit Patterns**


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x=df['number_emergency'], palette='Oranges_d', ax=ax)
ax.set_title('Emergency Admissions in Past Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Emergency Visits')
ax.set_ylabel('Patient Count')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x=df['inpatient_category'],
              order=['None', 'One', 'Few', 'Many'],
              palette='Purples_d', ax=ax)
ax.set_title('Prior Inpatient Visit Categories', fontsize=14, fontweight='bold')
ax.set_xlabel('Inpatient Category')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(x=df['had_outpatient'], palette=['#a8d8ea', '#aa96da'], ax=ax)
ax.set_title('Prior Outpatient Visits', fontsize=14, fontweight='bold')
ax.set_xlabel('Had Outpatient Visit')
ax.set_ylabel('Count')
ax.set_xticks([0, 1])
ax.set_xticklabels(['No', 'Yes'])
plt.tight_layout()
plt.show()


**Clinical and Diagnosis Risk Features**


**High-Risk Diagnosis Distribution**


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(x=df['any_high_risk_diagnosis'], palette=['#b2dfdb', '#ef9a9a'], ax=ax)
ax.set_title('High-Risk Diagnosis at Admission', fontsize=14, fontweight='bold')
ax.set_xlabel('High Risk Diagnosis Present')
ax.set_ylabel('Count')
ax.set_xticks([0, 1])
ax.set_xticklabels(['No', 'Yes'])
plt.tight_layout()
plt.show()


**Diabetes Medication Prescriptions**


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(x=df['diabetesMedication'], palette=['#ffe082', '#80cbc4'], ax=ax)
ax.set_title('Diabetes Medication Prescribed', fontsize=14, fontweight='bold')
ax.set_xlabel('On Diabetes Medication')
ax.set_ylabel('Count')
ax.set_xticks([0, 1])
ax.set_xticklabels(['No', 'Yes'])
plt.tight_layout()
plt.show()


**Insulin Usage Patterns**


In [ ]:
insulin_vars = ['insulin_steady', 'insulin_up', 'insulin_down', 'insulin_none']
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
palette = ['#81c784', '#e57373', '#64b5f6', '#ffb74d']

for ax, var, color in zip(axes.flatten(), insulin_vars, palette):
    sns.countplot(x=df[var], color=color, ax=ax)
    readmit_by_val = df.groupby(var)['readmitted_within_30'].mean()
    for patch, (val, rate) in zip(ax.patches, readmit_by_val.items()):
        ax.annotate(f'{rate:.1%}', (patch.get_x() + patch.get_width()/2, patch.get_height()),
                    ha='center', va='bottom', fontsize=9)
    ax.set_title(var.replace('_', ' ').title())
    ax.set_xlabel('Prescribed (1=Yes, 0=No)')
    ax.set_ylabel('Count')

plt.suptitle('Insulin Usage Patterns with Readmission Rates', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
df['insulin_used'] = (df['insulin_up'] | df['insulin_down']).astype(int)


In [ ]:
df.drop(columns=['insulin_up', 'insulin_down', 'insulin_steady'], inplace=True, errors='ignore')


**Lab Result Distribution**


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
a1c_dist  = df['A1Cresult'].value_counts().sort_index()
glu_dist  = df['max_glu_serum'].value_counts().sort_index()
x = np.arange(max(len(a1c_dist), len(glu_dist)))
ax.bar(x - 0.2, a1c_dist.reindex(range(4), fill_value=0), 0.4, label='A1C Result', color='#5c85d6')
ax.bar(x + 0.2, glu_dist.reindex(range(4), fill_value=0), 0.4, label='Max Glucose Serum', color='#e8a838')
ax.set_xlabel('Severity Level (0=None, 1=Normal, 2=Elevated, 3=High)')
ax.set_ylabel('Count')
ax.set_title('Lab Result Severity Distribution', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


**Administrative and Discharge Data**


**Admission Type Breakdown**


In [ ]:
admit_totals = df[['emergency_admission', 'urgent_admission',
                    'elective_admission', 'trauma_admission']].sum()
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=admit_totals.index, y=admit_totals.values, palette='coolwarm', ax=ax)
ax.set_title('Admission Type Counts', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


**Length of Hospital Stay**


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(x=df['time_in_hospital'], palette='viridis', ax=ax)
ax.set_title('Length of Hospital Stay (Days)', fontsize=14, fontweight='bold')
ax.set_xlabel('Days in Hospital')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(x=df['discharged_home'], palette=['#ef5350', '#26a69a'], ax=ax)
ax.set_title('Discharge Destination: Home vs Other', fontsize=14, fontweight='bold')
ax.set_xlabel('Discharged to Home')
ax.set_ylabel('Count')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Other Destination', 'Home'])
plt.tight_layout()
plt.show()


In [ ]:
left_ama = df['left_against_advice'].value_counts()
print(left_ama)


### EDA Summary — Key Cost Signals

- **Class imbalance confirmed**: 11.1% positive rate requires explicit imbalance handling
- **Demographic patterns**: Elderly patients (60+) are the highest-volume and highest-cost readmission segment
- **Discharge destination**: Clear split between home-discharge (higher risk) and facility-discharge (lower risk) — a direct cost intervention lever
- **Prior utilization**: Patients with multiple prior inpatient visits are dramatically overrepresented in readmissions
- **Lab instability**: Elevated A1C and glucose levels at admission correlate with higher readmission rates
- **Medication complexity**: High medication counts and mid-stay medication changes signal unstable discharge conditions


In [ ]:
print(df.columns.tolist())


### Feature Selection — Step 1: Binary Feature Impact Ranking


In [ ]:
from scipy import stats

binary_cols = [
    col for col in df.columns
    if df[col].nunique() == 2
    and col not in ['readmitted_within_30', 'gender', 'change']
]

binary_impact = []
for feat in binary_cols:
    try:
        rates = df.groupby(feat)['readmitted_within_30'].mean() * 100
        binary_impact.append({
            'Feature':    feat,
            'Count_1':    int(df[feat].sum()),
            'Pct_1':      round(df[feat].mean() * 100, 2),
            'Rate_0':     round(rates.get(0, 0), 2),
            'Rate_1':     round(rates.get(1, 0), 2),
            'Delta':      round(abs(rates.get(1, 0) - rates.get(0, 0)), 2)
        })
    except Exception as exc:
        print(f'Skipped {feat}: {exc}')

binary_impact_df = pd.DataFrame(binary_impact).sort_values('Delta', ascending=False).reset_index(drop=True)
display(binary_impact_df)

top10 = binary_impact_df.head(10)
x = np.arange(len(top10))
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - 0.2, top10['Rate_0'], 0.4, label='Feature = 0', color='#42a5f5')
ax.bar(x + 0.2, top10['Rate_1'], 0.4, label='Feature = 1', color='#ef5350')
ax.set_xticks(x)
ax.set_xticklabels(top10['Feature'], rotation=40, ha='right')
ax.set_ylabel('30-Day Readmission Rate (%)')
ax.set_title('Top 10 Binary Features by Readmission Rate Impact', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


#### Binary Feature Evaluation

For each binary feature, the analysis measures:
- **Prevalence**: What percentage of patients have value = 1
- **Readmission rate difference**: How much the readmission rate changes between patients with value 0 vs. 1

Features with the largest absolute difference in readmission rates between their two groups are the most actionable for cost-reduction targeting — they identify subpopulations where the presence or absence of a characteristic meaningfully changes readmission probability.


In [ ]:
low_value_cols = [
    'still_patient', 'patient_expired', 'discharged_within_institution',
    'discharged_for_outpatient', 'neonate_transfer', 'newborn_admission',
    'provider_referral', 'insulin_steady', 'insulin_up', 'insulin_down',
    'insulin_none', 'gender',
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
    'glipizide-metformin', 'glyburide-metformin', 'metformin-rosiglitazone',
    'metformin-pioglitazone', 'glimepiride-pioglitazone',
]
df.drop(columns=low_value_cols, inplace=True, errors='ignore')



Binary features removed at this stage:
- **Administrative artifacts**: `still_patient`, `patient_expired` — not predictive of 30-day discharge outcomes
- **Redundant medication indicators**: Already captured in summary features
- **Low-signal features**: Gender binary and others with near-zero readmission rate difference

Removing these reduces model complexity without losing predictive power.


In [ ]:
binary_cols = [col for col in df.columns
               if df[col].nunique() == 2 and col != 'readmitted_within_30']
print(binary_cols)


#### Binary Feature Re-Identification Post-Cleanup

After removing low-signal binary columns, the remaining binary features (exactly 2 unique values, excluding the target) are re-identified as the final candidate set for model training.


In [ ]:
numeric_cols = [
    col for col in df.select_dtypes(include=['int64', 'float64']).columns
    if col != 'readmitted_within_30' and df[col].nunique() > 2
]

numeric_impact = []
for feat in numeric_cols:
    try:
        grp = df.groupby('readmitted_within_30')[feat].mean()
        m0, m1 = grp.get(0, 0), grp.get(1, 0)
        corr = df[feat].corr(df['readmitted_within_30'])
        g0 = df.loc[df['readmitted_within_30']==0, feat]
        g1 = df.loc[df['readmitted_within_30']==1, feat]
        _, p = stats.ttest_ind(g0, g1, equal_var=False)
        numeric_impact.append({
            'Feature':      feat,
            'Mean_No':      round(m0, 3),
            'Mean_Yes':     round(m1, 3),
            'Abs_Diff':     round(abs(m1-m0), 3),
            'Corr':         round(corr, 4),
            'P_Value':      round(p, 4)
        })
    except Exception as exc:
        print(f'Skipped {feat}: {exc}')

numeric_impact_df = (
    pd.DataFrame(numeric_impact)
    .assign(AbsCorr=lambda d: d['Corr'].abs())
    .sort_values('AbsCorr', ascending=False)
    .drop(columns='AbsCorr')
    .reset_index(drop=True)
)
display(numeric_impact_df)

top_num = numeric_impact_df.head(10)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#1565c0' if c > 0 else '#b71c1c' for c in top_num['Corr']]
ax.barh(top_num['Feature'], top_num['Corr'], color=colors)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Correlation with 30-Day Readmission')
ax.set_title('Numerical Feature Correlations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()



For numerical features, the analysis evaluates:
- **Group means**: Average feature value for readmitted vs. non-readmitted patients
- **Absolute and percentage differences**: Quantify how much the means diverge
- **Point-biserial correlation**: Linear association with the binary target
- **T-test significance**: Statistical confidence that the group difference is not by chance

Features that show large mean differences AND statistical significance (p < 0.05) are prioritized as cost-relevant predictors.


In [ ]:
redundant_numeric = ['inpatient_ordinal', 'num_procedures', 'medications_used', 'age_ordinal']
df.drop(columns=redundant_numeric, inplace=True, errors='ignore')



Numerical features removed due to low signal or redundancy with engineered features:
- `inpatient_ordinal`: Information already captured in the categorical `inpatient_category` and `number_inpatient`
- `num_procedures`: Weak independent correlation with readmission after controlling for other factors
- `medications_used`: Aggregated count — less precise than the raw `num_medications`
- `age_ordinal`: Age signal better captured by the interaction feature `age_inpatient_interaction`


In [ ]:
numeric_cols = [col for col in df.select_dtypes(include=['int64','float64']).columns
                if col != 'readmitted_within_30']
print(numeric_cols)


In [ ]:
cat_cols = [col for col in df.select_dtypes(include='object').columns
            if col != 'readmitted_within_30']

cat_impact = []
for feat in cat_cols:
    try:
        contingency = pd.crosstab(df[feat], df['readmitted_within_30'])
        chi2, p, _, _ = stats.chi2_contingency(contingency)
        n = contingency.values.sum()
        r, k = contingency.shape
        phi2 = chi2 / n
        phi2c = max(0, phi2 - (k-1)*(r-1)/(n-1))
        rc = r - (r-1)**2/(n-1)
        kc = k - (k-1)**2/(n-1)
        cramers_v = np.sqrt(phi2c / min(kc-1, rc-1)) if min(kc-1, rc-1) > 0 else 0
        rates = df.groupby(feat)['readmitted_within_30'].mean() * 100
        cat_impact.append({
            'Feature':   feat,
            'Chi2':      round(chi2, 2),
            'P_Value':   round(p, 4),
            'CramersV':  round(cramers_v, 4),
            'MaxRate':   round(rates.max(), 2),
            'MinRate':   round(rates.min(), 2),
            'RateDiff':  round(rates.max() - rates.min(), 2)
        })
    except Exception as exc:
        print(f'Skipped {feat}: {exc}')

cat_impact_df = (
    pd.DataFrame(cat_impact)
    .sort_values('CramersV', ascending=False)
    .reset_index(drop=True)
)
display(cat_impact_df)


#### Categorical Feature Association Analysis

For categorical features, statistical association with the readmission target is measured using:
- **Chi-Square test**: Tests whether category distributions differ significantly between readmitted and non-readmitted patients
- **Cramér's V**: Effect size measure (0–1) indicating the strength of association

Readmission rates are also visualized per category to identify which specific values within each feature drive the highest cost risk.


In [ ]:
drop_cats = ['change', 'payer_code', 'metformin', 'medical_specialty']
df.drop(columns=drop_cats, inplace=True, errors='ignore')
cat_cols = ['gender', 'race', 'age']



Categorical features finalized for model input:
- **Retained**: `gender`, `race`, `age` — all show meaningful Cramér's V associations and align with known healthcare disparity patterns
- **Dropped**: `change`, `payer_code`, `metformin`, `medical_specialty` — weak association scores or information already captured in engineered features


In [ ]:
cat_cols = [col for col in df.select_dtypes(include='object').columns
            if col != 'readmitted_within_30']
print(cat_cols)


In [ ]:
model_features = list(pd.Index(binary_cols + numeric_cols + cat_cols).unique())
dup_cols = df.columns[df.columns.duplicated()].tolist()
print('Duplicate columns found:', dup_cols)



Binary, numerical, and categorical feature lists are merged into a single `final_features` list. Duplicate columns are detected and removed — any column appearing twice would bias model training by double-weighting its contribution.


In [ ]:
keep_cols = model_features + ['readmitted_within_30']
df = df[keep_cols].copy()


In [ ]:
df.to_csv(r'Data\cleaned_data.csv', index=False)
print(f'Saved cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns')


#### Final Dataset for Modeling

The cleaned and feature-engineered dataset is finalized by selecting only the columns in `final_features` plus the target `readmitted_within_30`. This is the exact input the models will train and evaluate on.


In [ ]:
df.head()


In [ ]:
print(df.columns.tolist())


### Selected Feature Set — Cost-Risk Signal Summary

The features entering the model represent four categories of cost-relevant signal:

- **Prior Utilization**: `number_inpatient`, `inpatient_category`, `number_emergency` — past resource use predicts future cost burden
- **Discharge Risk**: `discharged_home`, `discharged_to_facility`, `time_in_hospital` — discharge conditions shape readmission likelihood
- **Treatment Complexity**: `diabetesMedication`, `medication_changed`, `insulin_used`, `num_medications` — unstable management at discharge drives preventable returns
- **Clinical Severity**: `high_risk_diabetes`, `any_high_risk_diagnosis`, `A1Cresult`, `max_glu_serum`, `num_lab_procedures` — patient acuity determines baseline risk


In [ ]:
target = 'readmitted_within_30'
feature_cols = [c for c in df.columns if c != target]
X = df[feature_cols]
y = df[target]
print(f'Feature matrix: {X.shape} | Target distribution:\n{y.value_counts()}')


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
print(f'Train: {X_train.shape}, Test: {X_test.shape}')



The dataset is split 80/20 for train/test evaluation. `stratify=y` ensures the 11.1% positive rate is preserved in both splits — without this, a random split could produce a test set with significantly different class proportions and misleading evaluation metrics.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

binary_cols = list(binary_cols)

def make_preprocessor(scale=True):
    num_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        *([('scale', StandardScaler())] if scale else [])
    ])
    cat_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    bin_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent'))
    ])
    return ColumnTransformer(transformers=[
        ('num', num_pipe, numeric_cols),
        ('cat', cat_pipe, cat_cols),
        ('bin', bin_pipe, binary_cols),
    ])


#### Preprocessing Pipeline Architecture

The `build_preprocessor()` function constructs a `ColumnTransformer` with three parallel branches:
- **Numerical**: Mean imputation + optional `StandardScaler` (required for Logistic Regression, not needed for tree models)
- **Categorical**: Most-frequent imputation + `OneHotEncoder` with `handle_unknown='ignore'`
- **Binary**: Most-frequent imputation only — no encoding needed

Wrapping this in an `sklearn.Pipeline` guarantees that the preprocessing is fit only on training data and applied to test data — no data leakage.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score


## **Model Training and Evaluation**


### Logistic Regression


In [ ]:
lr_base = Pipeline([
    ('prep', make_preprocessor(scale=True)),
    ('clf',  LogisticRegression(max_iter=1000, random_state=42))
])
lr_base.fit(X_train, y_train)
y_pred_log    = lr_base.predict(X_test)
train_score_log = lr_base.score(X_train, y_train)
test_score_log  = accuracy_score(y_test, y_pred_log)
cv_score_log    = cross_val_score(lr_base, X_train, y_train, cv=5, scoring='accuracy')
print(f'LR Base  | Train: {train_score_log:.4f} | Test: {test_score_log:.4f} | CV: {cv_score_log.mean():.4f}')
print(classification_report(y_test, y_pred_log))


##### Baseline Logistic Regression

A standard `LogisticRegression` pipeline is trained first as a cost-free performance floor:
- Evaluation includes training accuracy, test accuracy, 5-fold cross-validation, and classification report
- The baseline will have high accuracy (~89%) but near-zero recall on class 1 — because the default model optimizes for majority class

This baseline exposes the core challenge: standard accuracy metrics are useless here. A model that never flags a readmission can score 88.9% accuracy while failing entirely at the actual business objective.


In [ ]:
lr_balanced = Pipeline([
    ('prep', make_preprocessor(scale=True)),
    ('clf',  LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
lr_balanced.fit(X_train, y_train)



The balanced variant uses `class_weight='balanced'` — this scales the loss function so the model treats each true positive in the minority class as 9x more important than a true negative in the majority class.

This is the minimum viable approach for a cost-reduction use case where missed high-risk patients represent thousands of dollars in avoidable readmission costs.


In [ ]:
train_acc_lr = lr_balanced.score(X_train, y_train)
y_pred_lr    = lr_balanced.predict(X_test)
test_acc_lr  = accuracy_score(y_test, y_pred_lr)
print(f'LR Balanced | Train: {train_acc_lr:.4f} | Test: {test_acc_lr:.4f}')


In [ ]:
print(classification_report(y_test, y_pred_lr))


In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
print('Confusion Matrix (LR Balanced):\n', cm_lr)


In [ ]:
scores_lr = cross_val_score(lr_balanced, X, y, cv=5, scoring='accuracy')
print(f'CV Scores: {scores_lr} | Mean: {scores_lr.mean():.4f}')



Comparing balanced vs. baseline Logistic Regression:
- Accuracy drops slightly (balanced model accepts more false positives)
- Recall on class 1 increases substantially — more readmitted patients are caught
- This tradeoff is exactly what a cost-reduction model should make


### Random Forest


In [ ]:
rf_base = Pipeline([
    ('prep', make_preprocessor(scale=False)),
    ('clf',  RandomForestClassifier(n_estimators=100, random_state=42))
])
rf_base.fit(X_train, y_train)
y_pred_rf_base = rf_base.predict(X_test)
train_score_rf = rf_base.score(X_train, y_train)
test_score_rf  = accuracy_score(y_test, y_pred_rf_base)
cv_score_rf    = cross_val_score(rf_base, X_train, y_train, cv=5, scoring='accuracy')
print(f'RF Base | Train: {train_score_rf:.4f} | Test: {test_score_rf:.4f} | CV: {cv_score_rf.mean():.4f}')
print(classification_report(y_test, y_pred_rf_base))


In [ ]:
rf_balanced = Pipeline([
    ('prep', make_preprocessor(scale=False)),
    ('clf',  RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])
rf_balanced.fit(X_train, y_train)


In [ ]:
train_acc_rf = rf_balanced.score(X_train, y_train)
y_pred_rf    = rf_balanced.predict(X_test)
test_acc_rf  = accuracy_score(y_test, y_pred_rf)
print(f'RF Balanced | Train: {train_acc_rf:.4f} | Test: {test_acc_rf:.4f}')


In [ ]:
print(classification_report(y_test, y_pred_rf))


In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
print('Confusion Matrix (RF Balanced):\n', cm_rf)


In [ ]:
cv_scores_rf = cross_val_score(rf_balanced, X, y, cv=5, scoring='accuracy')
print(f'CV Scores: {cv_scores_rf} | Mean: {cv_scores_rf.mean():.4f}')



Two Random Forest configurations are compared:

**Baseline Random Forest**
- Default settings, no imbalance handling
- High accuracy, poor minority class recall

**Balanced Random Forest**
- `class_weight='balanced'` applied
- Improves readmission detection at the cost of some precision

Both are evaluated on training accuracy, test accuracy, classification report, confusion matrix, and 5-fold CV. Random Forest is particularly useful here as a comparison against boosting methods — it handles feature interactions well but tends to be outperformed by gradient boosting on tabular imbalanced data.


### XGBoost


In [ ]:
from xgboost import XGBClassifier

xgb_base = Pipeline([
    ('prep', make_preprocessor(scale=False)),
    ('clf',  XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0))
])
xgb_base.fit(X_train, y_train)
y_pred_xgb_base = xgb_base.predict(X_test)
train_acc_xgb   = xgb_base.score(X_train, y_train)
test_acc_xgb    = accuracy_score(y_test, y_pred_xgb_base)
print(f'XGB Base | Train: {train_acc_xgb:.4f} | Test: {test_acc_xgb:.4f}')
print(classification_report(y_test, y_pred_xgb_base))


In [ ]:
class_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb_balanced = Pipeline([
    ('prep', make_preprocessor(scale=False)),
    ('clf',  XGBClassifier(eval_metric='logloss', scale_pos_weight=class_ratio,
                           random_state=42, verbosity=0))
])
xgb_balanced.fit(X_train, y_train)


XGBoost addresses class imbalance through `scale_pos_weight` — set to the ratio of negative-to-positive samples (~8:1 for this dataset).

This tells XGBoost that each true positive is 8x more valuable to get right than a true negative, directly aligning the optimization objective with the cost-reduction goal.


In [ ]:
train_acc_xgb_bal = xgb_balanced.score(X_train, y_train)
y_pred_xgb        = xgb_balanced.predict(X_test)
test_acc_xgb_bal  = accuracy_score(y_test, y_pred_xgb)
print(f'XGB Balanced | Train: {train_acc_xgb_bal:.4f} | Test: {test_acc_xgb_bal:.4f}')


In [ ]:
print(classification_report(y_test, y_pred_xgb))


In [ ]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print('Confusion Matrix (XGB Balanced):\n', cm_xgb)


In [ ]:
cv_scores_xgb_bal = cross_val_score(xgb_balanced, X, y, cv=5, scoring='accuracy')
print(f'CV Scores: {cv_scores_xgb_bal} | Mean: {cv_scores_xgb_bal.mean():.4f}')



### XGBoost Pipeline and Evaluation

The XGBoost pipeline integrates the preprocessor and classifier without scaling (tree models are scale-invariant).

Evaluation metrics:
- **Training/Test Accuracy**: Sanity check for overfitting
- **Classification Report**: Precision, recall, F1 per class — recall on class 1 is the primary business metric
- **Confusion Matrix**: Visual breakdown of correct vs. incorrect predictions
- **5-Fold CV**: Confirms the model generalizes beyond the specific train/test split

XGBoost is expected to outperform Logistic Regression and match or beat Random Forest on recall — its gradient boosting architecture is particularly effective on imbalanced tabular datasets.


### LightGBM


In [ ]:
from lightgbm import LGBMClassifier

lgb_base = Pipeline([
    ('prep', make_preprocessor(scale=False)),
    ('clf',  LGBMClassifier(random_state=42, verbose=-1))
])
lgb_base.fit(X_train, y_train)
y_pred_lgb_base  = lgb_base.predict(X_test)
train_acc_lgb_base = lgb_base.score(X_train, y_train)
test_acc_lgb_base  = accuracy_score(y_test, y_pred_lgb_base)
print(f'LGB Base | Train: {train_acc_lgb_base:.4f} | Test: {test_acc_lgb_base:.4f}')
print(classification_report(y_test, y_pred_lgb_base))


In [ ]:
lgb_balanced = Pipeline([
    ('prep', make_preprocessor(scale=False)),
    ('clf',  LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1))
])
lgb_balanced.fit(X_train, y_train)


In [ ]:
train_acc_lgb_bal = lgb_balanced.score(X_train, y_train)
y_pred_lgb_bal    = lgb_balanced.predict(X_test)
test_acc_lgb_bal  = accuracy_score(y_test, y_pred_lgb_bal)
print(f'LGB Balanced | Train: {train_acc_lgb_bal:.4f} | Test: {test_acc_lgb_bal:.4f}')


In [ ]:
print(classification_report(y_test, y_pred_lgb_bal))


In [ ]:
cm_lgb = confusion_matrix(y_test, y_pred_lgb_bal)
print('Confusion Matrix (LGB Balanced):\n', cm_lgb)


In [ ]:
cv_scores_lgb_bal = cross_val_score(lgb_balanced, X, y, cv=5, scoring='accuracy')
print(f'CV Scores: {cv_scores_lgb_bal} | Mean: {cv_scores_lgb_bal.mean():.4f}')


LightGBM is evaluated in two variants:

---

**1. LightGBM Baseline**
- No explicit imbalance handling
- Establishes LightGBM's natural performance ceiling before adjustments

---

**2. LightGBM Balanced**
- `class_weight='balanced'` applied
- Expected to deliver the best recall among all evaluated models
- LightGBM's leaf-wise growth strategy is particularly well-suited to the high-dimensional, sparse feature space created by one-hot encoding

---

**Evaluation Metrics (both variants)**
- Training and test accuracy
- Full classification report with per-class recall
- Confusion matrix
- 5-fold CV for generalization confirmation

LightGBM's speed advantage also makes it the practical choice for production deployment — faster inference means lower per-prediction operational costs.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

model_registry = [
    ('LR Base',           y_pred_log,      train_score_log,  test_score_log,  cv_score_log.mean()),
    ('LR Balanced',       y_pred_lr,       train_acc_lr,     test_acc_lr,     scores_lr.mean()),
    ('RF Base',           y_pred_rf_base,  train_score_rf,   test_score_rf,   cv_score_rf.mean()),
    ('RF Balanced',       y_pred_rf,       train_acc_rf,     test_acc_rf,     cv_scores_rf.mean()),
    ('XGB Base',          y_pred_xgb_base, train_acc_xgb,    test_acc_xgb,    None),
    ('XGB Balanced',      y_pred_xgb,      train_acc_xgb_bal,test_acc_xgb_bal,cv_scores_xgb_bal.mean()),
    ('LGB Base',          y_pred_lgb_base, train_acc_lgb_base,test_acc_lgb_base, None),
    ('LGB Balanced',      y_pred_lgb_bal,  train_acc_lgb_bal, test_acc_lgb_bal, cv_scores_lgb_bal.mean()),
]

model_summary = pd.DataFrame([{
    'Model':         name,
    'Train Acc':     round(tr, 4),
    'Test Acc':      round(te, 4),
    'CV Acc':        round(cv, 4) if cv else 'N/A',
    'Precision(1)':  round(precision_score(y_test, yp, zero_division=0), 4),
    'Recall(1)':     round(recall_score(y_test, yp, zero_division=0), 4),
    'F1(1)':         round(f1_score(y_test, yp, zero_division=0), 4),
} for name, yp, tr, te, cv in model_registry])

display(model_summary)


### Baseline Model Comparison

**Note**: Class 1 = 30-day readmission (the costly minority class).

---

### Summary of Findings

- **Default models achieve high accuracy but zero cost-reduction value** — they never flag a readmission
- **Balanced models sacrifice overall accuracy but dramatically improve class-1 recall** — this is the correct tradeoff for a cost-containment use case
- **Best recall on class 1**: Logistic Regression, XGBoost, and LightGBM balanced variants are competitive
- **LightGBM (balanced) selected for tuning**: Best combination of recall, speed, and interpretability


In [ ]:
lgb_clf = lgb_balanced.named_steps['clf']
feat_names_out = lgb_balanced.named_steps['prep'].get_feature_names_out()
feat_importance = pd.DataFrame({
    'Feature':    feat_names_out,
    'Importance': lgb_clf.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_importance['Feature'][:20][::-1],
        feat_importance['Importance'][:20][::-1], color='#5c85d6')
ax.set_xlabel('Feature Importance Score')
ax.set_title('Top 20 Features - LightGBM Balanced', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

display(feat_importance.head(20))


In [ ]:
import shap

lgb_clf = lgb_balanced.named_steps['clf']
X_test_prep = lgb_balanced.named_steps['prep'].transform(X_test)
shap_explainer = shap.TreeExplainer(lgb_clf)
shap_vals = shap_explainer.shap_values(X_test_prep)


In [ ]:
feat_names_out = lgb_balanced.named_steps['prep'].get_feature_names_out()
shap.summary_plot(shap_vals, X_test_prep, feature_names=feat_names_out)



Two interpretability methods are applied to the tuned LightGBM model:

---

**1. LightGBM Built-in Feature Importance**
- Extracts `.feature_importances_` from the trained classifier
- Plots the top 20 features ranked by split gain
- Fast and useful for identifying which features the model uses most frequently

---

**2. SHAP Summary Plot**
- `shap.TreeExplainer` computes exact SHAP values for each test-set prediction
- Each dot = one patient; position on x-axis = direction and magnitude of feature's impact on that patient's readmission prediction
- Color = feature value (red = high, blue = low)

SHAP values are critical for healthcare deployment: they let clinical staff understand *why* a patient was flagged as high-risk — not just *that* they were flagged. This audit trail is essential for clinical trust and regulatory compliance.


---

### Reading the SHAP Summary Plot

- **Y-axis**: Top 20 features ranked by mean absolute SHAP value
- **X-axis**: SHAP value — magnitude and direction of feature's impact on prediction
  - **Positive SHAP**: Feature pushed the prediction toward readmission (class 1) — cost risk signal
  - **Negative SHAP**: Feature pushed the prediction away from readmission

- **Color (feature value)**:
  - **Red** = high feature value
  - **Blue** = low feature value

For example: a red dot for `number_inpatient` with high positive SHAP value means "high prior inpatient visits strongly increase this patient's predicted readmission risk" — which aligns with clinical intuition and makes the model's logic auditable.


## **Hyperparameter Tuning**


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

lr_tuned_pipe = Pipeline([
    ('prep', make_preprocessor(scale=True)),
    ('clf',  LogisticRegression(class_weight='balanced', max_iter=1000,
                                solver='liblinear', random_state=42))
])
lr_param_grid = {
    'clf__C':       loguniform(0.001, 10),
    'clf__penalty': ['l1', 'l2']
}
lr_search = RandomizedSearchCV(
    lr_tuned_pipe, lr_param_grid, n_iter=20, scoring='f1',
    cv=4, random_state=42, verbose=1, n_jobs=-1
)
lr_search.fit(X_train, y_train)


In [ ]:
print('Best LR params:', lr_search.best_params_)
y_pred_lr_tuned = lr_search.best_estimator_.predict(X_test)
print(f'Train Acc: {accuracy_score(y_train, lr_search.best_estimator_.predict(X_train)):.4f}')
print(f'Test Acc:  {accuracy_score(y_test, y_pred_lr_tuned):.4f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_lr_tuned))


In [ ]:
print(classification_report(y_test, y_pred_lr_tuned))


In [ ]:
from xgboost import XGBClassifier
from scipy.stats import uniform, randint

class_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb_tuned_pipe = Pipeline([
    ('prep', make_preprocessor(scale=False)),
    ('clf',  XGBClassifier(eval_metric='logloss', scale_pos_weight=class_ratio,
                           random_state=42, verbosity=0))
])
xgb_param_grid = {
    'clf__n_estimators':    randint(100, 300),
    'clf__learning_rate':   uniform(0.01, 0.2),
    'clf__max_depth':       randint(3, 10),
    'clf__subsample':       uniform(0.6, 0.4),
    'clf__colsample_bytree':uniform(0.6, 0.4),
    'clf__min_child_weight':randint(1, 10)
}
xgb_search = RandomizedSearchCV(
    xgb_tuned_pipe, xgb_param_grid, n_iter=30, scoring='f1',
    cv=4, random_state=42, verbose=1, n_jobs=-1
)
xgb_search.fit(X_train, y_train)


In [ ]:
print('Best XGB params:', xgb_search.best_params_)
y_pred_xgb_tuned = xgb_search.best_estimator_.predict(X_test)
print(f'Train Acc: {accuracy_score(y_train, xgb_search.best_estimator_.predict(X_train)):.4f}')
print(f'Test Acc:  {accuracy_score(y_test, y_pred_xgb_tuned):.4f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_xgb_tuned))


In [ ]:
print(classification_report(y_test, y_pred_xgb_tuned))


In [ ]:
from scipy.stats import randint, uniform

lgb_param_grid = {
    'clf__n_estimators':     randint(100, 800),
    'clf__learning_rate':    uniform(0.01, 0.2),
    'clf__max_depth':        randint(3, 15),
    'clf__num_leaves':       randint(20, 100),
    'clf__min_child_samples':randint(10, 100),
    'clf__subsample':        uniform(0.5, 0.5),
    'clf__colsample_bytree': uniform(0.5, 0.5)
}


In [ ]:
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

strat_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
lgb_search = RandomizedSearchCV(
    lgb_balanced, lgb_param_grid,
    n_iter=30, scoring='f1', cv=strat_cv,
    n_jobs=-1, verbose=1, random_state=42
)
lgb_search.fit(X_train, y_train)


In [ ]:
print('Best LGB params:', lgb_search.best_params_)
final_model = lgb_search.best_estimator_
y_pred_lgb_tuned = final_model.predict(X_test)
print(f'Train Acc: {accuracy_score(y_train, final_model.predict(X_train)):.4f}')
print(f'Test Acc:  {accuracy_score(y_test, y_pred_lgb_tuned):.4f}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_lgb_tuned))


In [ ]:
print(classification_report(y_test, y_pred_lgb_tuned))



`RandomizedSearchCV` is used to tune all three classifiers, optimizing for **F1-score** — the harmonic mean of precision and recall. F1 is the right optimization target here because it forces the model to balance catching readmissions (recall) against the operational cost of false alarms (precision).

---

**1. Logistic Regression**
- Tuned: `C` (regularization strength), `penalty` (L1 vs L2)
- `class_weight='balanced'` retained
- 4-fold stratified CV used for scoring

---

**2. XGBoost**
- Tuned: `n_estimators`, `learning_rate`, `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`
- `scale_pos_weight` retained for imbalance handling
- 4-fold stratified CV

---

**3. LightGBM**
- Tuned: `num_leaves`, `min_child_samples`, `learning_rate`, `feature_fraction`, `bagging_fraction`
- 3-fold StratifiedKFold for efficiency at scale
- Best parameters compared against balanced baseline to confirm improvement


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def score_model(name, estimator, X_tr, y_tr, X_te, y_te):
    return {
        'Model':        name,
        'Train Acc':    round(accuracy_score(y_tr,  estimator.predict(X_tr)), 4),
        'Test Acc':     round(accuracy_score(y_te,  estimator.predict(X_te)), 4),
        'Precision(1)': round(precision_score(y_te, estimator.predict(X_te), zero_division=0), 4),
        'Recall(1)':    round(recall_score(y_te,    estimator.predict(X_te), zero_division=0), 4),
        'F1(1)':        round(f1_score(y_te,        estimator.predict(X_te), zero_division=0), 4),
    }

tuned_models = [
    ('LR Tuned',  lr_search.best_estimator_),
    ('XGB Tuned', xgb_search.best_estimator_),
    ('LGB Tuned', lgb_search.best_estimator_),
]
tuned_results = pd.DataFrame([
    score_model(n, m, X_train, y_train, X_test, y_test) for n, m in tuned_models
])
display(tuned_results)


In [ ]:
def extract_feature_names(preprocessor):
    names = []
    for _, transformer, cols in preprocessor.transformers_:
        if hasattr(transformer, 'get_feature_names_out'):
            names.extend(transformer.get_feature_names_out(cols))
        else:
            names.extend(cols)
    return names

lgb_clf     = lgb_search.best_estimator_.named_steps['clf']
prep_lgb    = lgb_search.best_estimator_.named_steps['prep']
feat_names  = extract_feature_names(prep_lgb)

feat_imp_df = pd.DataFrame({
    'Feature':    feat_names,
    'Importance': lgb_clf.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df.head(20),
            palette='Blues_r', ax=ax)
ax.set_title('Top 20 Features - LightGBM (Tuned)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
display(feat_imp_df.head(20))


In [ ]:
import shap

X_test_prep   = prep_lgb.transform(X_test)
shap_explainer = shap.TreeExplainer(lgb_clf)
shap_vals      = shap_explainer.shap_values(X_test_prep)
shap.summary_plot(shap_vals, features=X_test_prep, feature_names=feat_names)


## **SMOTE — Synthetic Minority Oversampling**


Preprocessing is applied only to `X_train` before SMOTE — fitting the preprocessor on `X_test` would cause data leakage.


In [ ]:
data_preprocessor = make_preprocessor(scale=False)
X_train_prep = data_preprocessor.fit_transform(X_train)
X_test_prep  = data_preprocessor.transform(X_test)


In [ ]:
from imblearn.over_sampling import SMOTE

oversampler = SMOTE(random_state=42)
X_resampled, y_resampled = oversampler.fit_resample(X_train_prep, y_train)


In [ ]:
from collections import Counter
print('Before SMOTE:', Counter(y_train))
print('After  SMOTE:', Counter(y_resampled))


SMOTE successfully balances the training set — the minority class (readmitted patients) is now equally represented.

The test set remains untouched at its natural 11.1% positive rate, ensuring evaluation reflects real-world deployment conditions.


In [ ]:
from lightgbm import LGBMClassifier

smote_lgb = LGBMClassifier(random_state=42, verbose=-1)
smote_lgb.fit(X_resampled, y_resampled)


In [ ]:
smote_preds = smote_lgb.predict(X_test_prep)
print(classification_report(y_test, smote_preds))


In [ ]:
readmit_probs = smote_lgb.predict_proba(X_test_prep)[:, 1]
train_acc_smote = accuracy_score(y_resampled, smote_lgb.predict(X_resampled))
print(f'Training Accuracy (SMOTE): {train_acc_smote:.4f}')

decision_thresholds = [0.10, 0.15, 0.20]
for thresh in decision_thresholds:
    preds_t = (readmit_probs >= thresh).astype(int)
    print(f'\n--- Threshold = {thresh} ---')
    print(classification_report(y_test, preds_t))
    print(f'Accuracy: {accuracy_score(y_test, preds_t):.4f}')
    print(f'Confusion Matrix:\n{confusion_matrix(y_test, preds_t)}')



SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic training examples for the minority class by interpolating between existing minority-class samples in feature space.

Unlike random oversampling (which duplicates existing records), SMOTE creates new, plausible patient profiles — this forces the model to learn more general decision boundaries rather than memorizing repeated examples.

**Trade-off**: SMOTE dramatically improves recall at the cost of precision. In a cost-reduction context, the high-recall variant is most useful for initial patient triage — clinical judgment and follow-up resources determine which flagged patients receive intervention.


In [ ]:
import joblib
joblib.dump(final_model, 'tuned_model.joblib')
print('Model saved to tuned_model.joblib')


In [ ]:
import joblib
loaded_model = joblib.load('tuned_model.joblib')
print('Model loaded successfully:', type(loaded_model))
